# 字符串与结构化数组

学习目标：批量处理字符串，用合适的 dtype 保存文本和混合字段记录，判断截断、存储与共享修改的边界。

前置知识：dtype、索引、类型转换、视图与副本、Python 字符串与对象引用。

运行环境：Python 3.12、NumPy 2.5；StringDType 与 numpy.strings 从 NumPy 2.0 开始提供，本章按 2.5 行为编写。

环境准备：[环境配置与运行](README.md)。

工作目录：本 Notebook 所在目录；重启内核后从上到下运行。

示例使用单元内构造的数据，后续单元沿用首次导入的 np；选学内容用于理解记录的内存布局。

## 1 清理字符串数组

三个设备编号含有多余空格和不一致的大小写。把它们放入一维数组后，用 np.strings.strip 去掉两端空白，再用 np.strings.upper 转为大写。每个位置仍对应原来的一个设备。

numpy.strings 按元素处理字符串数组。下面两个操作产生结果数组，不改写原始编号；np.strings.str_len 用于检查每项长度。

In [1]:
import numpy as np

raw = np.array([" ab-1 ", " Cd-20", "ef-3  "])
clean = np.strings.upper(np.strings.strip(raw))

print(clean)  # ['AB-1' 'CD-20' 'EF-3']
print(clean.shape, clean.dtype)  # (3,)；仍是 Unicode 字符串数组。
print(np.strings.str_len(clean))  # [4 5 4]
print(raw)  # 原来的空格与大小写保留。

['AB-1' 'CD-20' 'EF-3']
(3,) <U6
[4 5 4]
[' ab-1 ' ' Cd-20' 'ef-3  ']


## 2 定长字符串与截断

普通字符串序列默认创建定长 Unicode 数组，容量由创建时最长的输入决定。dtype 中 U 后面的整数是每个元素最多保存的 Unicode 码点数；每个码点占 4 字节。例如 U3 每项占 12 字节。

容量是 dtype 的一部分，后来赋入更长文本不会自动扩容，超出的部分会被截断。创建时显式指定过小的容量也会截断。这里用普通汉字和 ASCII 字母演示，不把码点数等同于所有文字的屏幕显示宽度。

In [2]:
names = np.array(["甲", "乙组"])
names[0] = "甲组加长"
limited = np.array(["ABCD", "中文标签"], dtype="U3")

print(names, names.dtype)  # ['甲组' '乙组']，容量仍为 U2。
print(limited)  # ['ABC' '中文标']，创建时已截断。
print(limited.itemsize, limited.nbytes)  # 12 24
print(np.strings.str_len(limited))  # [3 3]

['甲组' '乙组'] <U2
['ABC' '中文标']
12 24
[3 3]


定长字节串用 S 表示，S3 的容量是 3 字节，不是 3 个 Unicode 码点。字节串没有自动解释文本编码的能力；下面先用 ASCII 字节观察截断，b 前缀表示 Python 字节串字面量。

定长容量必须在写入前决定。将已截断的数组转换到更大的 dtype，只会扩大后续存储空间，不会恢复丢失的内容。

In [3]:
codes = np.array([b"ABCD", b"xy"], dtype="S3")
codes[1] = b"HELLO"
wider = codes.astype("S8")

print(codes, codes.dtype)  # [b'ABC' b'HEL']，dtype 为 |S3。
print(codes.itemsize, codes.nbytes)  # 3 6
print(wider)  # 仍是 [b'ABC' b'HEL']，丢失的后缀不会恢复。
print(np.strings.str_len(codes))  # [3 3]，本次统计字节数。

[b'ABC' b'HEL'] |S3
3 6
[b'ABC' b'HEL']
[3 3]


文本与字节之间用明确的编码转换：np.strings.encode 把文本编码成字节，np.strings.decode 按相同编码还原文本。下面“中文”含 2 个码点，UTF-8 编码后占 6 字节。

如果按字节容量截断，可能截在一个多字节字符内部。严格解码会报错，不能用忽略错误来证明数据完整。

In [4]:
text = np.array(["中文", "A"])
encoded = np.strings.encode(text, encoding="utf-8")

print(encoded.dtype)  # |S6
print(np.strings.str_len(text), np.strings.str_len(encoded))  # [2 1] [6 1]
print(np.strings.decode(encoded, encoding="utf-8"))  # ['中文' 'A']

truncated = encoded.astype("S4")
try:
    np.strings.decode(truncated, encoding="utf-8", errors="strict")
except UnicodeDecodeError as error:
    print(type(error).__name__)  # UnicodeDecodeError：第二个汉字的字节不完整。
else:
    raise AssertionError("预期截断后的 UTF-8 字节无法完整解码")

|S6
[2 1] [6 1]
['中文' 'A']
UnicodeDecodeError


## 3 可变长度字符串

如果名称长度不能提前确定，可显式使用 np.dtypes.StringDType()。它保存可变长度的 UTF-8 字符串，不要求所有元素预留相同的文本容量；不会因为初始名称较短，就把后续长名称按原长度截断。

StringDType 与固定的 U、S 是不同表示。下面继续使用 numpy.strings 处理文本，但不依赖创建时的最大长度。

In [5]:
names = np.array(["甲", "乙组"], dtype=np.dtypes.StringDType())
names[0] = "甲组加长"

print(names)  # ['甲组加长' '乙组']，长名称完整保留。
print(names.shape, names.dtype)  # (2,) StringDType()
print(np.strings.str_len(names))  # [4 2]
print(names.astype("U2"))  # ['甲组' '乙组']：转回容量不足的定长类型仍会截断。

['甲组加长' '乙组']
(2,) StringDType()
[4 2]
['甲组' '乙组']


StringDType 默认把非字符串输入转换为字符串。如果“设备名称必须原本就是字符串”是输入约束，可用 coerce=False 拒绝隐式转换；它不会把数值输入悄悄变成文本。

In [6]:
converted = np.array(["A", 12], dtype=np.dtypes.StringDType())
print(converted)  # ['A' '12']，数字已变成字符串。

try:
    np.array(["A", 12], dtype=np.dtypes.StringDType(coerce=False))
except ValueError as error:
    print(type(error).__name__)  # ValueError：严格模式不接受数值输入。
else:
    raise AssertionError("预期严格 StringDType 拒绝非字符串")

['A' '12']
ValueError


StringDType 的主 ndarray 数据缓冲区用于保存管理字符串存储位置的元数据，不能像定长 U 或 S 那样把它直接当作文本内容。完整文本使用 UTF-8 表示，但不是简单地依次放在主数组缓冲区中。

因此，nbytes 不能作为 StringDType 全部文本及相关存储的总内存统计。下面只比较主缓冲区的计数；不读取内部地址或假定其私有布局。

In [7]:
names = np.array(["甲"], dtype=np.dtypes.StringDType())
before = names.nbytes
names[0] = "汉" * 100

print(np.strings.str_len(names))  # [100]，字符串明显增长。
print(before, names.nbytes)  # 当前环境中都为 16；这是主缓冲区的大小。
print(before == names.nbytes)  # True，不能据此推断总内存没有变化。

[100]
16 16
True


## 4 保存混合字段记录

一条设备记录同时包含名称、浮点读数和布尔状态。结构化 dtype 为每个字段分别指定名称和类型；每个数组元素都是一条具有相同字段结构的记录，而不是把整张表转成字符串。

下面有 3 条记录，数组形状为 (3,)。字段 name 使用 U4，value 使用 float64，valid 使用 bool。创建记录时使用元组，元组中的值按 dtype 的字段顺序填写。读数单位为摄氏度。

In [8]:
record_dtype = np.dtype([
    ("name", "U4"),
    ("value", np.float64),
    ("valid", np.bool_),
])
records = np.array([
    ("北区", 20.5, True),
    ("南区", 19.0, False),
    ("东区", 22.0, True),
], dtype=record_dtype)

print(records)
print(records.shape, records.dtype.names)  # (3,) ('name', 'value', 'valid')
print(records["value"], records["value"].dtype)  # [20.5 19. 22.] float64
print(records[records["valid"]]["name"])  # ['北区' '东区']

[('北区', 20.5,  True) ('南区', 19. , False) ('东区', 22. ,  True)]
(3,) ('name', 'value', 'valid')
[20.5 19.  22. ] float64
['北区' '东区']


按字段名读取，例如 records['value']，得到共享原数组存储的视图；对这个字段视图赋值会修改对应记录。字段数量不会自动成为数组的第二个轴。

下面继续使用前面的 records，把第一条记录的读数修正为 21.0，并核对名称和状态字段仍对应原记录。字段里的 U4 仍有定长限制，放入更长名称同样会截断。

In [9]:
values = records["value"]
values[0] = 21.0
records["name"][1] = "南区备用站"

print(values.shape, values.dtype)  # (3,) float64
print(np.shares_memory(values, records))  # True
print(records)  # 第一条读数为 21.0；第二条名称截为“南区备用”。
print(records["valid"])  # [ True False True]

(3,) float64
True
[('北区', 21.,  True) ('南区备用', 19., False) ('东区', 22.,  True)]
[ True False  True]


一条记录的内存由各字段占据的字节组成。dtype.fields 记录字段的类型及其相对于记录起点的字节偏移，itemsize 是整条记录的字节数。

默认 align=False 时，各字段紧密排列。继续查看 records 的布局：U4 占 16 字节，float64 占 8 字节，bool 占 1 字节；每条记录共 25 字节。整个结构有统一的 dtype，各字段的类型可以不同。

In [10]:
for name in records.dtype.names:
    field_dtype, offset = records.dtype.fields[name]
    print(name, field_dtype, offset)
# name 从偏移 0 开始，value 从 16 开始，valid 从 24 开始。
print(records.itemsize, records.nbytes)  # 25 75：3 条记录。
print(records["value"].strides)  # (25,)：相邻读数间隔一整条记录。

name <U4 0
value float64 16
valid bool 24
25 75
(25,)


## 5 object 数组与浅复制

object 数组也能装不同类型的内容，但每个槽位保存的是 Python 对象的引用，没有结构化 dtype 那样的固定字段定义。nbytes 只统计数组槽位占用，不递归计算被引用的列表、字典等对象。

下面明确创建两个槽位，分别放入列表和字典。copy 复制数组的引用槽位，不递归复制 Python 对象，称为浅复制。因此数组缓冲区不共享，也不代表内部可变对象彼此独立。

In [11]:
objects = np.empty(2, dtype=object)
objects[0] = [1, 2]
objects[1] = {"valid": True}
copied = objects.copy()
copied[0].append(3)

print(objects[0], copied[0])  # 两边都是 [1, 2, 3]。
print(objects[0] is copied[0])  # True：引用同一个列表。
print(np.shares_memory(objects, copied))  # False：数组引用槽位已经复制。
print(objects.shape, objects.dtype)  # (2,) object
print(objects.nbytes == objects.size * objects.itemsize)  # True，只统计槽位。

copied[1] = {"valid": False}
print(objects[1], copied[1])  # 替换副本槽位后分别为 True、False。

[1, 2, 3] [1, 2, 3]
True
False
(2,) object
True
{'valid': True} {'valid': False}


## 6 选学：多字段选择

结构化数组用字段名列表选择多个字段时，得到的仍是视图。选出字段的顺序可以改变，但字段在记录中的原始偏移以及记录的 itemsize 会保留，未选字段只是不再显示。这和用整数数组选择记录的高级索引不同。

下面每条记录原本占 12 字节；只选 id 与 value 后，中间字段的 4 字节位置仍留在布局里。需要把字段重新紧密排列时，可用 numpy.lib.recfunctions.repack_fields。它在需要改变布局时生成副本；已经符合目标布局时可以直接返回输入。

In [12]:
from numpy.lib.recfunctions import repack_fields

records = np.array([(1, 90, 2.5), (2, 80, 3.5)],
                   dtype=[("id", "i4"), ("reserved", "i4"), ("value", "f4")])
selected = records[["id", "value"]]
packed = repack_fields(selected)

print(selected.dtype)  # 偏移仍为 0、8，itemsize 仍为 12。
print(selected.shape, np.shares_memory(selected, records))  # (2,) True
print(packed.dtype, packed.itemsize)  # 两字段紧密排列，itemsize 为 8。
print(np.shares_memory(packed, records))  # False：本例确实重新排列并复制。
selected["value"][0] = 9.5
print(records["value"], packed["value"])  # [9.5 3.5] 与 [2.5 3.5]。

{'names': ['id', 'value'], 'formats': ['<i4', '<f4'], 'offsets': [0, 8], 'itemsize': 12}
(2,) True
[('id', '<i4'), ('value', '<f4')] 8
False
[9.5 3.5] [2.5 3.5]


## 7 选学：对齐与子数组字段

align=True 可以在字段之间及记录末尾加入填充，使字段偏移满足各类型的对齐要求。这样得到的布局不保证与任意 C 编译器的结构体完全一致；对接外部格式仍应核对偏移和记录长度。

下面只比较一个 1 字节标记和一个 4 字节整数的布局，不把更大的记录长度当作更多有效字段。

In [13]:
fields = [("flag", "u1"), ("count", "i4")]
packed_dtype = np.dtype(fields)
aligned_dtype = np.dtype(fields, align=True)

print(packed_dtype.fields["count"][1], packed_dtype.itemsize)  # 1 5
print(aligned_dtype.fields["count"][1], aligned_dtype.itemsize)  # 4 8
print(aligned_dtype.isalignedstruct)  # True

1 5
4 8
True


字段还可以包含固定形状的子数组。下面每条记录存一个编号和二维位置，position 字段的两个分量依次表示 x、y 坐标，单位为米。

数组本身含 3 条记录，形状为 (3,)；访问 position 时，子数组的形状 (2,) 追加到记录数组形状后，得到 (3, 2)。这适合每条记录内部结构固定的情况。

In [14]:
point_dtype = np.dtype([("id", "i4"), ("position", "f8", (2,))])
points = np.array([(1, [0.0, 1.0]), (2, [2.0, 3.0]), (3, [4.0, 5.0])],
                  dtype=point_dtype)
positions = points["position"]

print(points.shape, positions.shape)  # (3,) (3, 2)
print(positions.dtype, np.shares_memory(positions, points))  # float64 True
positions[0, 1] = 1.5
print(points["position"][0])  # [0. 1.5]，字段视图的修改写回第一条记录。

(3,) (3, 2)
float64 True
[0.  1.5]


## 本章小结

（1）numpy.strings 按元素处理文本。定长 U 的容量按 Unicode 码点计算，S 按字节计算；超出容量会截断，扩宽已截断数组不能找回内容。

（2）StringDType 保存可变长度文本，但主缓冲区不能按定长文本解释，nbytes 也不是全部文本存储的统计。

（3）结构化 dtype 为每条记录定义同一组字段。字段访问通常是共享存储的视图，字段类型、偏移与 itemsize 共同描述记录布局。

（4）object 数组保存 Python 对象引用，copy 是浅复制。多字段选择会保留原偏移；需要紧密布局时再考虑重新打包。

## 练习

（1）清理三个标签的两端空格并转为大写，打印结果及每项长度，再选择长度超过 3 的标签。保留原始输入并核对它没有被修改。

In [15]:
labels = np.array([" ab ", "cdef ", " Gh "])

# 在此完成清理和筛选；检查清理结果形状为 (3,)，筛选结果形状为 (1,)。
# 分别打印原始标签和新数组。

（2）先预测以下两个数组赋值后的内容与 dtype，再运行核对。现在要求以后接收任意长度的设备名称，不能截断，应选哪种表示？若外部格式明确限制最多 4 个码点，又应在写入前增加什么检查？说明选择理由。

In [16]:
fixed = np.array(["A", "BB"], dtype="U2")
variable = np.array(["A", "BB"], dtype=np.dtypes.StringDType())
fixed[0] = "LONG"
variable[0] = "LONG"

print(fixed, fixed.dtype)
print(variable, variable.dtype)
# 写下预测、选择理由，以及按码点数检查长度的代码。

['LO' 'BB'] <U2
['LONG' 'BB'] StringDType()


（3）把以下输入创建为结构化数组：name 字段为 U4，count 字段为 int32，valid 字段为 bool。选出有效记录的名称，并通过 count 字段视图把第一条计数改为 5。核对形状、字段 dtype 和共享修改。

In [17]:
rows = [("甲组", 2, True), ("乙组", 3, False), ("丙组", 4, True)]

# 在此定义 dtype 并创建数组；每条记录使用给定元组。
# 检查数组形状为 (3,)，修改后第一条 count 为 5，其他字段保持原值。

（4）先预测副本中列表的 append 是否会影响原数组，再运行核对。随后把副本的第一个槽位替换为新列表 [9]，再次打印两边。为什么这两种修改的影响不同？为什么 shares_memory 不能独自回答列表是否共享？

In [18]:
source = np.empty(1, dtype=object)
source[0] = [1]
copied = source.copy()
copied[0].append(2)

print(source[0], copied[0])
print(np.shares_memory(source, copied))
# 在此替换 copied[0]，重新打印两边，并用 is 检查对象身份。

[1, 2] [1, 2]
False


## 参考与引用来源

| 网站 | 本章参考内容与定位 |
| --- | --- |
| NumPy 官方文档（numpy.org） | NumPy 2.5，核查日期：2026-09-20。[String functionality](https://numpy.org/doc/2.5/reference/routines.strings.html) 的说明与 Note：numpy.strings 模块及版本；[Working with Arrays of Strings And Bytes](https://numpy.org/doc/2.5/user/basics.strings.html) 的 Fixed-width data types、Variable-width strings、Coercing Non-strings、Casting To and From Fixed-Width Strings：定长容量、截断、StringDType、UTF-8 与主缓冲区区别；[strip](https://numpy.org/doc/2.5/reference/generated/numpy.strings.strip.html)、[upper](https://numpy.org/doc/2.5/reference/generated/numpy.strings.upper.html)、[str_len](https://numpy.org/doc/2.5/reference/generated/numpy.strings.str_len.html) 的定义和 Parameters：逐项清理及码点／字节长度；[encode](https://numpy.org/doc/2.5/reference/generated/numpy.strings.encode.html)、[decode](https://numpy.org/doc/2.5/reference/generated/numpy.strings.decode.html)：按指定编码转换；[Structured arrays](https://numpy.org/doc/2.5/user/basics.rec.html) 的 Structured datatype creation、Automatic byte offsets and alignment、Accessing Individual Fields、Accessing Multiple Fields、repack_fields：混合记录、字段视图、子数组、偏移、对齐和重新打包；[Glossary：object array](https://numpy.org/doc/2.5/glossary.html#term-object-array)：Python 对象引用；[copy 的 Notes](https://numpy.org/doc/2.5/reference/generated/numpy.copy.html#notes) 与 [ndarray.copy 的 Examples](https://numpy.org/doc/2.5/reference/generated/numpy.ndarray.copy.html#examples)：object 浅复制；[ndarray.nbytes](https://numpy.org/doc/2.5/reference/generated/numpy.ndarray.nbytes.html) 的定义、Notes、Examples：元素存储统计范围；[shares_memory](https://numpy.org/doc/2.5/reference/generated/numpy.shares_memory.html) 的定义、Returns：数组缓冲区的共享检查。 |
| Python 官方文档（docs.python.org） | Python 3.12，[Built-in Types：str.encode](https://docs.python.org/3.12/library/stdtypes.html#str.encode)、[bytes.decode](https://docs.python.org/3.12/library/stdtypes.html#bytes.decode)：文本编码、字节解码与严格错误处理。 |